# SMS Spam Classifier

**Goal:** build a machine learning model that decides whether a text message is *spam* or *ham* (a normal message).

**Dataset:** the [SMS Spam Collection](https://archive.ics.uci.edu/dataset/228/sms+spam+collection) from the UCI Machine Learning Repository, about 5,500 labelled English SMS messages.

**Approach:** turn text into numbers with TF-IDF, train Naive Bayes and Logistic Regression, then compare them with metrics that suit an imbalanced problem (precision, recall and F1, not just accuracy).

In [ ]:
import io
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, classification_report,
                             f1_score, precision_score, recall_score)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42

## 1. Load the data

In [ ]:
UCI_URL = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
MIRROR_URL = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
COLUMNS = ["label", "message"]


def load_sms_data():
    """Download the dataset from UCI, falling back to a public mirror."""
    try:
        with urllib.request.urlopen(UCI_URL, timeout=30) as response:
            archive = zipfile.ZipFile(io.BytesIO(response.read()))
        with archive.open("SMSSpamCollection") as f:
            return pd.read_csv(f, sep="\t", header=None, names=COLUMNS, encoding="latin-1")
    except Exception as error:
        print(f"UCI download failed ({error}); trying the mirror...")
    return pd.read_csv(MIRROR_URL, sep="\t", header=None, names=COLUMNS, encoding="latin-1")


df = load_sms_data()
print(df.shape)
df.head()

## 2. Explore the data

Before modelling, check the class balance. If 87% of messages are ham, a model that always says "ham" scores 87% accuracy while catching zero spam. That's why accuracy alone is misleading here.

In [ ]:
counts = df["label"].value_counts()
print(counts)
print(f"\nSpam share: {counts['spam'] / len(df):.1%}")

df["length"] = df["message"].str.len()
print("\nAverage message length (characters):")
print(df.groupby("label")["length"].mean().round(1))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
counts.plot(kind="bar", ax=axes[0], color=["#4C78A8", "#E45756"], rot=0)
axes[0].set_title("Messages per class")
axes[0].set_ylabel("Count")
for label, color in [("ham", "#4C78A8"), ("spam", "#E45756")]:
    axes[1].hist(df.loc[df["label"] == label, "length"], bins=30, alpha=0.65,
                 label=label, color=color)
axes[1].set_title("Message length by class")
axes[1].set_xlabel("Characters")
axes[1].legend()
plt.tight_layout()
plt.show()

## 3. Train/test split

An 80/20 split, **stratified** so both sets keep the same spam proportion. The test set stays untouched until evaluation.

In [ ]:
X = df["message"]
y = (df["label"] == "spam").astype(int)  # 1 = spam, 0 = ham

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f"Train: {len(X_train)} messages | Test: {len(X_test)} messages")
print(f"Spam share  train: {y_train.mean():.1%}  test: {y_test.mean():.1%}")

## 4. Model 1: TF-IDF + Multinomial Naive Bayes

- **TF-IDF** turns each message into a vector. Words that are frequent in a message but rare overall (like "prize" or "claim") get higher weight.
- **Naive Bayes** is a fast, classic text classifier that works well on small datasets. It assumes words are independent given the class, which is not true, but the model still performs surprisingly well.

Both steps sit inside a `Pipeline`, so the vectoriser is only ever fitted on training data (no data leakage).

In [ ]:
def evaluate(name, model, X_test, y_test):
    """Print and return the key metrics for the spam class."""
    predictions = model.predict(X_test)
    scores = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions),
        "Recall": recall_score(y_test, predictions),
        "F1": f1_score(y_test, predictions),
    }
    print(f"=== {name} ===")
    print(classification_report(y_test, predictions, target_names=["ham", "spam"], digits=3))
    return scores


nb_model = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, stop_words="english", ngram_range=(1, 2), min_df=2)),
    ("clf", MultinomialNB(alpha=0.1)),
])
nb_model.fit(X_train, y_train)
nb_scores = evaluate("Naive Bayes", nb_model, X_test, y_test)

ConfusionMatrixDisplay.from_estimator(
    nb_model, X_test, y_test, display_labels=["ham", "spam"], cmap="Blues", colorbar=False
)
plt.title("Naive Bayes: confusion matrix")
plt.show()

**Reading the confusion matrix:**
- *False positives* (ham flagged as spam) mean a real message goes missing. That's usually the **costlier** mistake, so precision matters.
- *False negatives* (spam that slips through) are annoying but less harmful, so recall matters too.

## 5. Model 2: Logistic Regression, and a comparison

In [ ]:
lr_model = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, stop_words="english", ngram_range=(1, 2), min_df=2)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
])
lr_model.fit(X_train, y_train)
lr_scores = evaluate("Logistic Regression", lr_model, X_test, y_test)

results = pd.DataFrame([nb_scores, lr_scores]).set_index("Model")
print("Summary on the test set")
results.round(4)

## 6. Is the result reliable? Cross-validation

A single train/test split can be lucky or unlucky. 5-fold stratified cross-validation trains and tests on five different splits, so we can see how stable the F1 score is.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
for name, model in [("Naive Bayes", nb_model), ("Logistic Regression", lr_model)]:
    f1_scores = cross_val_score(model, X, y, cv=cv, scoring="f1")
    print(f"{name:20s} F1 = {f1_scores.mean():.3f} +/- {f1_scores.std():.3f}")

## 7. What did the model learn?

Naive Bayes stores how likely each word is in spam versus ham. The words with the largest gap are its strongest spam signals.

In [ ]:
vectorizer = nb_model.named_steps["tfidf"]
classifier = nb_model.named_steps["clf"]
words = np.array(vectorizer.get_feature_names_out())

spam_signal = classifier.feature_log_prob_[1] - classifier.feature_log_prob_[0]
top_spam = words[np.argsort(spam_signal)[-15:][::-1]]
top_ham = words[np.argsort(spam_signal)[:15]]

print("Strongest SPAM words:", ", ".join(top_spam))
print("Strongest HAM words: ", ", ".join(top_ham))

## 8. Try it on new messages

In [ ]:
def classify(message, model=nb_model):
    """Return the predicted label and the model's spam probability."""
    spam_probability = model.predict_proba([message])[0][1]
    label = "SPAM" if spam_probability >= 0.5 else "ham"
    return label, spam_probability


examples = [
    "WINNER!! You have been selected to receive a 900 pound prize. Call now to claim!",
    "Hey, are we still meeting at 6 for dinner?",
    "Free entry in a weekly competition. Text WIN to 80086 to enter now",
    "Can you send me the notes from today's lecture?",
]
for text in examples:
    label, probability = classify(text)
    print(f"[{label:4s}] spam probability {probability:6.1%} | {text}")

## 9. Results and takeaways

> **Fill this in after running the notebook.** Replace the placeholders with your own numbers and observations.

- Best model: **[MODEL NAME]** with **[XX.X]%** F1 on the test set (precision **[XX.X]%**, recall **[XX.X]%**).
- Cross-validated F1: **[0.XXX ± 0.XXX]**, which shows [how stable the result is].
- Most telling spam words: **[LIST 3-4 FROM SECTION 7]**.

**Limitations**
- The dataset is small, English only and fairly old, so modern spam (links, crypto, WhatsApp scams) may be missed.
- The model reads words only. It ignores sender, links and message timing.
- It would need retraining as spam tactics change.

**Next steps:** try character n-grams, add hand-made features (digit count, currency symbols, ALL-CAPS ratio), tune the decision threshold to favour precision, and wrap the model in a small web app.